# NB_EXTRA_FIGS — two extra figures for the JFDS paper

**CPU-only, reads CSVs only.** Produces:
- **fig6_regime**  ← uses **NB6** results (`nb4_outputs/agent_perf/*_agent.csv` or `perf_checkpoints/*_perf.csv`)
- **fig7_stability** ← uses **NB5** results (`nb4_outputs/ext_checkpoints/*_armA.csv` or `ext_results.csv`)

Run Cell 1 (mount + locate), then Cell 2 (fig6, NB6) and Cell 3 (fig7, NB5).
Figures are saved to `nb4_outputs/figures_paper/`. Send me the two PDFs and I
drop them into the manuscript.


## Cell 1 — mount + locate the result files

In [1]:
from google.colab import drive; drive.mount("/content/drive")
import os, glob, numpy as np, pandas as pd
import matplotlib; import matplotlib.pyplot as plt
plt.rcParams.update({"font.family":"serif","axes.spines.top":False,"axes.spines.right":False,"savefig.bbox":"tight"})
ROOT="/content/drive/MyDrive/nb4_outputs"; FIG=os.path.join(ROOT,"figures_paper"); os.makedirs(FIG,exist_ok=True)
BLUE,RED,GREY,GREEN="#2c6fbb","#c0392b","#7f8c8d","#27ae60"
def savefig(fig,n): fig.savefig(os.path.join(FIG,n+".pdf")); fig.savefig(os.path.join(FIG,n+".png"),dpi=300); plt.close(fig); print("saved",n)
def _read(paths):
    fr=[pd.read_csv(p) for p in paths if os.path.exists(p)]
    return pd.concat(fr,ignore_index=True) if fr else None

# NB6 walk-forward per-market results (has 'window' + 'agent' + benchmark rows)
NB6 = _read(glob.glob(os.path.join(ROOT,"agent_perf","*_agent.csv"))) \
   or _read(glob.glob(os.path.join(ROOT,"perf_checkpoints","*_perf.csv")))
# NB5 arm-A results (three-arm, per-seed)
NB5 = None
c=os.path.join(ROOT,"ext_results.csv")
if os.path.exists(c):
    d=pd.read_csv(c); NB5=d[d.get("arm","A")=="A"] if "arm" in d else d
if NB5 is None:
    NB5=_read(glob.glob(os.path.join(ROOT,"ext_checkpoints","*_armA.csv")))
print("NB6 (walk-forward):", None if NB6 is None else f"{len(NB6)} rows, cols {list(NB6.columns)[:6]}...")
print("NB5 (arm A)       :", None if NB5 is None else f"{len(NB5)} rows, cols {list(NB5.columns)[:6]}...")

Mounted at /content/drive
NB6 (walk-forward): 60 rows, cols ['market', 'group', 'window', 'span', 'agent', 'state']...
NB5 (arm A)       : 434 rows, cols ['market', 'group', 'arm', 'agent', 'state', 'seed']...


## Cell 2 — fig6_regime  (NB6 data)

PPO minus buy-and-hold, per walk-forward window, pooled over markets. Shows the
policy tracks passive in bull windows and protects in bear/high-vol windows —
the 'economic value is bounded but regime-dependent' story.

In [2]:
assert NB6 is not None, "NB6 results not found — check agent_perf / perf_checkpoints on Drive"
P=NB6.copy()
scol="Sharpe" if "Sharpe" in P else "sharpe"
bh=P[(P.seed==-1)&(P.agent=="BuyHold")].set_index(["market","window"])[scol].rename("bh")
ppo=(P[(P.seed>=0)&(P.agent=="PPO")&(P.state=="Baseline")]
       .groupby(["market","window"])[scol].mean().rename("ppo"))
d=pd.concat([ppo,bh],axis=1).dropna(); d["gap"]=d["ppo"]-d["bh"]
g=d.groupby(level="window")["gap"]; m=g.mean(); se=g.std()/g.count()**0.5
fig,ax=plt.subplots(figsize=(5.4,3.6)); x=np.arange(len(m))
cols=[GREEN if v>0 else RED for v in m.values]
ax.bar(x,m.values,yerr=1.96*se.values,color=cols,alpha=.85,capsize=4)
ax.axhline(0,color="k",lw=0.8)
ax.set_xticks(x); ax.set_xticklabels([f"W{int(w)}" for w in m.index])
ax.set_xlabel("walk-forward window (chronological)"); ax.set_ylabel("PPO $-$ BuyHold  ($\\Delta$Sharpe)")
ax.set_title("Regime dependence: RL protects in down/high-vol windows",fontsize=10)
savefig(fig,"fig6_regime")
print(d.groupby(level="window")["gap"].mean().round(3).to_string())

saved fig6_regime
window
0   -0.212
1   -0.060
2    0.175
3    0.000


## Cell 3 — fig7_stability  (NB5 data)

Seed dispersion (SD of Sharpe across seeds) per agent family. Tabular Q-Learning
is far noisier than PPO — this justifies pooled inference and is a clean
methodological point for the Limitations.

In [3]:
assert NB5 is not None, "NB5 arm-A results not found — check ext_checkpoints / ext_results.csv"
P=NB5.copy(); scol="Sharpe" if "Sharpe" in P else "sharpe"
P=P[P.seed>=0]
disp=(P.groupby(["agent","market","state"])[scol].std()
        .groupby(level="agent").mean().sort_values())
fig,ax=plt.subplots(figsize=(5.0,3.2)); y=np.arange(len(disp))
ax.barh(y,disp.values,color=BLUE,alpha=.85)
ax.set_yticks(y); ax.set_yticklabels(disp.index)
ax.set_xlabel("mean seed dispersion (SD of Sharpe across seeds)")
ax.set_title("Training stability by agent family (lower = more stable)",fontsize=10)
for i,v in enumerate(disp.values): ax.text(v,i,f" {v:.2f}",va="center",fontsize=8)
savefig(fig,"fig7_stability")
print(disp.round(3).to_string())

saved fig7_stability
agent
PPO           0.062
Q-Learning    0.358
